# Proverb / Idiom Completion Evaluation

Evaluate proverb/idiom **completion** on:

1. **Open-weight ladder:** Qwen3.5 (0.8B → 2B → 4B) via Hugging Face
2. **Frontier APIs:** GPT-5.6, Claude Opus 5, Gemini 3.5 via [OpenRouter](https://openrouter.ai/)

- English (`data/english.json`) is the default control set.
- Point `DATA_PATH` at `data/hausa.json` when Hausa items are ready.
- Metrics: normalized exact match and soft (substring) match.

**Colab tip:** Runtime → Change runtime type → GPU (T4) for the Qwen ladder. OpenRouter frontier eval needs only an API key (CPU is fine).

## 1. Install dependencies

In [ ]:
%pip install -q "transformers>=4.51.0" accelerate sentencepiece bitsandbytes pandas matplotlib tqdm openai

## 2. Config

In [ ]:
import os
from pathlib import Path

# --- Paths ---
# In Colab: upload/clone the repo, then set REPO_ROOT accordingly.
# Example after `!git clone https://github.com/<you>/hausa-proverb-idioms.git`:
# REPO_ROOT = Path("/content/hausa-proverb-idioms")
REPO_ROOT = Path("..").resolve()  # notebook lives in notebooks/
if not (REPO_ROOT / "data" / "english.json").exists():
    # Fallback for Colab when notebook is opened from Drive or /content
    for candidate in [
        Path("/content/hausa-proverb-idioms"),
        Path("/content"),
        Path("."),
    ]:
        if (candidate / "data" / "english.json").exists():
            REPO_ROOT = candidate
            break

DATA_PATH = REPO_ROOT / "data" / "english.json"  # switch to hausa.json later
RESULTS_DIR = REPO_ROOT / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# --- Models (small → large) ---
MODEL_IDS = [
    "Qwen/Qwen3.5-0.8B",
    "Qwen/Qwen3.5-2B",
    "Qwen/Qwen3.5-4B",
]

# Approximate parameter counts (billions) for the scaling plot
MODEL_PARAMS_B = {
    "Qwen/Qwen3.5-0.8B": 0.8,
    "Qwen/Qwen3.5-2B": 2.0,
    "Qwen/Qwen3.5-4B": 4.0,
}

# --- Generation ---
MAX_NEW_TOKENS = 32
DO_SAMPLE = False  # greedy decoding for reproducibility

# Loading: "auto" | "float16" | "4bit"
# Use "4bit" on free Colab if 4B OOMs.
LOAD_MODE = "float16"

# Optional Hugging Face token for gated / rate-limited downloads
HF_TOKEN = os.environ.get("HF_TOKEN") or None

print(f"REPO_ROOT  = {REPO_ROOT}")
print(f"DATA_PATH  = {DATA_PATH} (exists={DATA_PATH.exists()})")
print(f"RESULTS_DIR= {RESULTS_DIR}")
print(f"Models     = {MODEL_IDS}")
print(f"LOAD_MODE  = {LOAD_MODE}")

## 3. Load dataset

In [ ]:
import json

with open(DATA_PATH, encoding="utf-8") as f:
    dataset = json.load(f)

assert isinstance(dataset, list) and len(dataset) > 0
for item in dataset:
    assert {"id", "prompt", "answer"} <= set(item.keys()), item

print(f"Loaded {len(dataset)} items from {DATA_PATH.name}")
print("Sample:", dataset[0])

## 4. Prompting, normalization, and scoring

In [ ]:
import re
import string

SYSTEM_PROMPT = (
    "You complete proverbs and idioms. "
    "Given an incomplete proverb or idiom, reply with ONLY the missing completion. "
    "Do not restate the prompt. Do not explain."
)


def build_user_prompt(fragment: str) -> str:
    return (
        f"Complete the following proverb or idiom.\n\n"
        f"Fragment: {fragment}\n\n"
        f"Completion:"
    )


def normalize_text(text: str) -> str:
    text = text.lower().strip()
    text = text.replace("…", " ").replace("...", " ")
    # Drop common wrappers
    text = text.strip("\"'`")
    # Remove punctuation
    table = str.maketrans("", "", string.punctuation)
    text = text.translate(table)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def exact_match(pred: str, gold: str) -> bool:
    return normalize_text(pred) == normalize_text(gold)


def soft_match(pred: str, gold: str) -> bool:
    p, g = normalize_text(pred), normalize_text(gold)
    if not p or not g:
        return False
    return g in p or p in g


# Quick self-check
assert exact_match("Saves nine!", "saves nine")
assert soft_match("saves nine, they say", "saves nine")
print("Scoring helpers OK")

## 5. Model load + generate helpers

In [ ]:
import gc
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig


def load_model_and_tokenizer(model_id: str, load_mode: str = "float16"):
    tok_kwargs = {"trust_remote_code": True}
    if HF_TOKEN:
        tok_kwargs["token"] = HF_TOKEN
    tokenizer = AutoTokenizer.from_pretrained(model_id, **tok_kwargs)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    model_kwargs = {
        "trust_remote_code": True,
        "device_map": "auto",
    }
    if HF_TOKEN:
        model_kwargs["token"] = HF_TOKEN

    if load_mode == "4bit":
        model_kwargs["quantization_config"] = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_compute_dtype=torch.float16,
        )
    elif load_mode == "float16":
        model_kwargs["torch_dtype"] = torch.float16
    else:
        model_kwargs["torch_dtype"] = "auto"

    model = AutoModelForCausalLM.from_pretrained(model_id, **model_kwargs)
    model.eval()
    return model, tokenizer


def unload_model(model):
    del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


@torch.inference_mode()
def complete_proverb(model, tokenizer, fragment: str) -> str:
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": build_user_prompt(fragment)},
    ]
    try:
        text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
        )
    except Exception:
        # Fallback if chat template is unavailable
        text = SYSTEM_PROMPT + "\n\n" + build_user_prompt(fragment)

    inputs = tokenizer(text, return_tensors="pt")
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    gen_kwargs = {
        "max_new_tokens": MAX_NEW_TOKENS,
        "do_sample": DO_SAMPLE,
        "pad_token_id": tokenizer.pad_token_id,
    }
    if not DO_SAMPLE:
        gen_kwargs["temperature"] = None
        gen_kwargs["top_p"] = None

    output_ids = model.generate(**inputs, **gen_kwargs)
    new_tokens = output_ids[0, inputs["input_ids"].shape[-1] :]
    completion = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
    # Keep first line only (models sometimes add explanations)
    completion = completion.splitlines()[0].strip() if completion else ""
    return completion


print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 6. Run evaluation across the model ladder

In [ ]:
from tqdm.auto import tqdm
import pandas as pd

all_rows = []
summary_rows = []

for model_id in MODEL_IDS:
    print("\n" + "=" * 60)
    print(f"Loading {model_id} ({LOAD_MODE})...")
    try:
        model, tokenizer = load_model_and_tokenizer(model_id, LOAD_MODE)
    except Exception as e:
        print(f"Failed to load {model_id}: {e}")
        print("Tip: set LOAD_MODE = '4bit' and retry, or skip this size.")
        continue

    n_exact = 0
    n_soft = 0
    for item in tqdm(dataset, desc=model_id.split("/")[-1]):
        pred = complete_proverb(model, tokenizer, item["prompt"])
        em = exact_match(pred, item["answer"])
        sm = soft_match(pred, item["answer"])
        n_exact += int(em)
        n_soft += int(sm)
        all_rows.append(
            {
                "model_id": model_id,
                "params_b": MODEL_PARAMS_B.get(model_id),
                "id": item["id"],
                "prompt": item["prompt"],
                "gold": item["answer"],
                "prediction": pred,
                "exact_match": em,
                "soft_match": sm,
            }
        )

    n = len(dataset)
    summary_rows.append(
        {
            "model_id": model_id,
            "params_b": MODEL_PARAMS_B.get(model_id),
            "n": n,
            "exact_match_acc": n_exact / n,
            "soft_match_acc": n_soft / n,
        }
    )
    print(
        f"{model_id}: exact={n_exact}/{n} ({n_exact/n:.1%}), "
        f"soft={n_soft}/{n} ({n_soft/n:.1%})"
    )

    unload_model(model)
    del tokenizer

predictions_df = pd.DataFrame(all_rows)
summary_df = pd.DataFrame(summary_rows)
display(summary_df)
predictions_df.head()

## 7. Scaling plot

In [ ]:
import matplotlib.pyplot as plt

if summary_df.empty:
    print("No results to plot.")
else:
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.plot(
        summary_df["params_b"],
        summary_df["exact_match_acc"],
        marker="o",
        label="Exact match",
    )
    ax.plot(
        summary_df["params_b"],
        summary_df["soft_match_acc"],
        marker="s",
        label="Soft match",
    )
    ax.set_xlabel("Parameters (B)")
    ax.set_ylabel("Accuracy")
    ax.set_title(f"Qwen3.5 scaling — {DATA_PATH.name}")
    ax.set_ylim(0, 1.05)
    ax.grid(True, alpha=0.3)
    ax.legend()
    for _, row in summary_df.iterrows():
        ax.annotate(
            row["model_id"].split("/")[-1],
            (row["params_b"], row["exact_match_acc"]),
            textcoords="offset points",
            xytext=(4, 6),
            fontsize=8,
        )
    plt.tight_layout()
    plot_path = RESULTS_DIR / f"scaling_{DATA_PATH.stem}.png"
    fig.savefig(plot_path, dpi=150)
    print(f"Saved plot → {plot_path}")
    plt.show()

## 8. Export results

In [ ]:
stem = DATA_PATH.stem
pred_path = RESULTS_DIR / f"predictions_{stem}.csv"
sum_path = RESULTS_DIR / f"summary_{stem}.csv"
json_path = RESULTS_DIR / f"summary_{stem}.json"

predictions_df.to_csv(pred_path, index=False)
summary_df.to_csv(sum_path, index=False)
summary_df.to_json(json_path, orient="records", indent=2)

print(f"Predictions → {pred_path}")
print(f"Summary CSV → {sum_path}")
print(f"Summary JSON → {json_path}")

# Optional: download from Colab
try:
    from google.colab import files  # type: ignore

    files.download(str(pred_path))
    files.download(str(sum_path))
except ImportError:
    pass

## 9. Frontier models via OpenRouter

Call frontier APIs through [OpenRouter](https://openrouter.ai/) (OpenAI-compatible chat completions):

| Display name | OpenRouter model id |
|--------------|---------------------|
| GPT-5.6 | `openai/gpt-5.6-sol` |
| Claude Opus 5 | `anthropic/claude-opus-5` |
| Gemini 3.5 | `google/gemini-3.5-flash` |

Set your key once (Colab Secrets, env var, or the cell below). Do **not** commit the key.

In [ ]:
import os
import time
from openai import OpenAI

# Prefer Colab Secrets / env; fall back to paste-here for local runs.
OPENROUTER_API_KEY = os.environ.get("OPENROUTER_API_KEY")
if not OPENROUTER_API_KEY:
    try:
        from google.colab import userdata  # type: ignore

        OPENROUTER_API_KEY = userdata.get("OPENROUTER_API_KEY")
    except Exception:
        OPENROUTER_API_KEY = None

if not OPENROUTER_API_KEY:
    # Paste temporarily if needed (clear cell output before sharing the notebook).
    OPENROUTER_API_KEY = ""  # e.g. "sk-or-v1-..."

assert OPENROUTER_API_KEY, (
    "Set OPENROUTER_API_KEY as an env var, Colab secret, or in this cell."
)

OPENROUTER_MODELS = [
    "openai/gpt-5.6-sol",          # GPT-5.6 (flagship Sol)
    "anthropic/claude-opus-5",     # Claude Opus 5
    "google/gemini-3.5-flash",     # Gemini 3.5
]

OPENROUTER_DISPLAY = {
    "openai/gpt-5.6-sol": "GPT-5.6",
    "anthropic/claude-opus-5": "Claude Opus 5",
    "google/gemini-3.5-flash": "Gemini 3.5",
}

or_client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=OPENROUTER_API_KEY,
    default_headers={
        "HTTP-Referer": "https://github.com/hausa-proverb-idioms",
        "X-Title": "Hausa Proverb Idiom Eval",
    },
)


def complete_proverb_openrouter(model_id: str, fragment: str, retries: int = 3) -> str:
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": build_user_prompt(fragment)},
    ]
    last_err = None
    for attempt in range(retries):
        try:
            resp = or_client.chat.completions.create(
                model=model_id,
                messages=messages,
                max_tokens=MAX_NEW_TOKENS,
                temperature=0,
            )
            content = (resp.choices[0].message.content or "").strip()
            return content.splitlines()[0].strip() if content else ""
        except Exception as e:
            last_err = e
            time.sleep(1.5 * (attempt + 1))
    raise RuntimeError(f"{model_id} failed after {retries} retries: {last_err}")


print("OpenRouter client ready.")
print("Models:", OPENROUTER_MODELS)

In [ ]:
frontier_rows = []
frontier_summary = []

for model_id in OPENROUTER_MODELS:
    display_name = OPENROUTER_DISPLAY.get(model_id, model_id)
    print("\n" + "=" * 60)
    print(f"OpenRouter → {display_name} ({model_id})")

    n_exact = 0
    n_soft = 0
    for item in tqdm(dataset, desc=display_name):
        try:
            pred = complete_proverb_openrouter(model_id, item["prompt"])
            err = None
        except Exception as e:
            pred = ""
            err = str(e)
            print(f"  error on {item['id']}: {err}")

        em = exact_match(pred, item["answer"]) if pred else False
        sm = soft_match(pred, item["answer"]) if pred else False
        n_exact += int(em)
        n_soft += int(sm)
        frontier_rows.append(
            {
                "backend": "openrouter",
                "model_id": model_id,
                "display_name": display_name,
                "id": item["id"],
                "prompt": item["prompt"],
                "gold": item["answer"],
                "prediction": pred,
                "exact_match": em,
                "soft_match": sm,
                "error": err,
            }
        )

    n = len(dataset)
    frontier_summary.append(
        {
            "backend": "openrouter",
            "model_id": model_id,
            "display_name": display_name,
            "n": n,
            "exact_match_acc": n_exact / n,
            "soft_match_acc": n_soft / n,
        }
    )
    print(
        f"{display_name}: exact={n_exact}/{n} ({n_exact/n:.1%}), "
        f"soft={n_soft}/{n} ({n_soft/n:.1%})"
    )

frontier_predictions_df = pd.DataFrame(frontier_rows)
frontier_summary_df = pd.DataFrame(frontier_summary)
display(frontier_summary_df)
frontier_predictions_df.head()

In [ ]:
# Bar chart: frontier vs open-weight ladder (soft match)
import matplotlib.pyplot as plt

compare_rows = []
if "summary_df" in globals() and not summary_df.empty:
    for _, row in summary_df.iterrows():
        compare_rows.append(
            {
                "label": row["model_id"].split("/")[-1],
                "family": "Qwen3.5 (local)",
                "soft_match_acc": row["soft_match_acc"],
                "exact_match_acc": row["exact_match_acc"],
            }
        )
for _, row in frontier_summary_df.iterrows():
    compare_rows.append(
        {
            "label": row["display_name"],
            "family": "Frontier (OpenRouter)",
            "soft_match_acc": row["soft_match_acc"],
            "exact_match_acc": row["exact_match_acc"],
        }
    )

compare_df = pd.DataFrame(compare_rows)
if compare_df.empty:
    print("No results to compare.")
else:
    fig, ax = plt.subplots(figsize=(9, 4))
    x = range(len(compare_df))
    colors = [
        "#4C78A8" if f == "Qwen3.5 (local)" else "#F58518"
        for f in compare_df["family"]
    ]
    ax.bar(x, compare_df["soft_match_acc"], color=colors)
    ax.set_xticks(list(x))
    ax.set_xticklabels(compare_df["label"], rotation=30, ha="right")
    ax.set_ylabel("Soft-match accuracy")
    ax.set_ylim(0, 1.05)
    ax.set_title(f"Open-weight vs frontier — {DATA_PATH.name}")
    ax.grid(True, axis="y", alpha=0.3)
    plt.tight_layout()
    compare_plot = RESULTS_DIR / f"compare_openrouter_{DATA_PATH.stem}.png"
    fig.savefig(compare_plot, dpi=150)
    print(f"Saved plot → {compare_plot}")
    plt.show()

# Export frontier results
stem = DATA_PATH.stem
f_pred = RESULTS_DIR / f"predictions_openrouter_{stem}.csv"
f_sum = RESULTS_DIR / f"summary_openrouter_{stem}.csv"
f_json = RESULTS_DIR / f"summary_openrouter_{stem}.json"
frontier_predictions_df.to_csv(f_pred, index=False)
frontier_summary_df.to_csv(f_sum, index=False)
frontier_summary_df.to_json(f_json, orient="records", indent=2)
print(f"Predictions → {f_pred}")
print(f"Summary CSV → {f_sum}")
print(f"Summary JSON → {f_json}")

## Next steps

1. Replace / expand `data/english.json` as needed.
2. Add `data/hausa.json` with the same `{id, prompt, answer}` schema.
3. Set `DATA_PATH` to the Hausa file and re-run the local ladder (sections 3–8) and/or OpenRouter (section 9).
4. Compare English vs Hausa to test whether scaling holds for low-resource figurative completion, and how frontier APIs compare to the open-weight ladder.